# **Imports**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision

# **Parameter initialization and data preparation**

In [2]:
# Parameters
batch_size = 8
num_epochs = 10
device = 'cuda:0'
num_classes = 10

# Load dataset
transform = torchvision.transforms.Compose(
    [torchvision.transforms.ToTensor(),
     torchvision.transforms.Normalize((0.5), (0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
valset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(trainset)

# Create dataloaders
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True)
valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                         shuffle=False)

100%|██████████| 170M/170M [00:03<00:00, 43.4MB/s]


Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )


In [3]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize to show images correctly
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

In [4]:
# Print some samples of dataset as a sanity check

# Get some random training images
dataiter = iter(trainloader)
example_images, example_labels = dataiter.next()

print(example_images.shape)

# Show images
imshow(torchvision.utils.make_grid(example_images))
# Print labels
print(' '.join('%5s' % classes[example_labels[j]] for j in range(batch_size)))

AttributeError: '_SingleProcessDataLoaderIter' object has no attribute 'next'

# **Define Models**

In [5]:
class DenseNet(nn.Module):
    def __init__(self, input_features, num_classes):
        # Instantiation of layers and creation of trainable parameters
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_features, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, num_classes)
        # >> Your code goes here <<
        # (You can copy the code from last lecture)

    def forward(self, x):
        # Forward pass: the computations that are done on the input -> output
        x = self.flatten(x)
        self.fc1(x)
        self.fc2(x)
        self.fc3(x)
        self.fc4(x)
        self.fc5(x)
        # >> Your code goes here <<
        # (You can copy the code from last lecture)
        return x

In [6]:
class ConvNet(nn.Module):
    def __init__(self, input_channels, num_classes):
        super().__init__()
        # >> Your code goes here <<

        # (You can copy the code from last lecture)

    def forward(self, x):
        # >> Your code goes here <<
        # (You can copy the code from last lecture)
        return x

# **Make a training loop**

In [ ]:
def compute_run_acc(logits, labels):
    _, pred = torch.max(logits.data, 1)
    return (pred == labels).sum().item()

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Instantiate model and optimizer
# RUN THIS CELL twice: once with "conv_net = False" and once "conv_net = True"

conv_net = False

if conv_net:
    model = ConvNet(3, num_classes).to(device)
    print("Convolutional model loaded")
    tr_accuracies_conv = np.zeros(num_epochs)
    val_accuracies_conv = np.zeros(num_epochs)
else:
    model = DenseNet(1024*3, num_classes).to(device)
    print("Dense model loaded")
    tr_accuracies_dense = np.zeros(num_epochs)
    val_accuracies_dense = np.zeros(num_epochs)
print("Number of trainable parameters: {}".format(count_parameters(model)))

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
best_val_acc = 0

start_time = time.time()

for epoch_nr in range(num_epochs):

    print("Epoch {}:".format(epoch_nr))

    # Train model
    running_loss = 0.0
    running_acc = 0.0

    # Train model
    # >> Your code goes here <<
    # You can re-use code from previous lab session

    # Print results
    tr_acc = 100 * running_acc/len(trainloader.dataset)
    print('>> TRAIN: Epoch {} completed | tr_loss: {:.4f} | tr_acc: {:.2f}%'.format(
        epoch_nr, running_loss/len(trainloader.dataset), tr_acc))

    # Get validation results
    # >> Your code goes here <<
    # You can re-use code from previous lab session
    # (beware the variablename testloader has been changed in valloader)

    val_acc = 100 * running_acc/len(valloader.dataset)
    print('>> VALIDATION: Epoch {} | val_acc: {:.2f}%'.format(epoch_nr, val_acc))

    if conv_net:
        tr_accuracies_conv[epoch_nr] = tr_acc
        val_accuracies_conv[epoch_nr] = val_acc
    else:
        tr_accuracies_dense[epoch_nr] = tr_acc
        val_accuracies_dense[epoch_nr] = val_acc

    # Save model if best accuracy on validation dataset until now
    # >> Your code goes here <<
    # You can re-use code from previous lab session

end_time = time.time()
print('Finished Training in {:.2f} seconds'.format(end_time-start_time))

# **Investigate results**

In [ ]:
plt.figure()
plt.plot(tr_accuracies_dense, label='Dense network')
plt.plot(tr_accuracies_conv, label='Conv network')
plt.title('Training results')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend()

plt.figure()
plt.plot(val_accuracies_dense, label='Dense network')
plt.plot(val_accuracies_conv, label='Conv network')
plt.title('Validation results')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend()
plt.show()

In [ ]:
# Check some predictions from trained model

# Get some random validation images
dataiter = iter(valloader)
example_images, example_labels = dataiter.next()

# Show images
imshow(torchvision.utils.make_grid(example_images))
logits = model(example_images.to(device))
_, pred_classes = torch.max(logits.data, 1)

# Print labels
print('GT labels  :' + ' '.join('%5s' % classes[example_labels[j].item()] for j in range(batch_size)))
print('Predictions:' + ' '.join('%5s' % classes[pred_classes[j].item()] for j in range(batch_size)))